In [3]:
import pandas as pd
import numpy as np

## خوندن دیتاست خام


In [8]:
df = pd.read_excel("First Dataset.xlsx")

print(df.shape)
df.head()

(61, 17)


,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,F,19.0,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53.0,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3
2,1003,Parsa,F,31.0,Shiraz,Fars,2023-06-20,Gold,21,59.46,1248.66,22,Online Wallet,iPhone,Yes,6,1
3,1004,Sina,F,58.0,Mashhad,Khorasan,2021-11-08,Gold,23,266.15,6121.45,40,Card,Android,No,4,4
4,1005,Kimia,M,28.0,Isfahan,Isfahan,2021-10-21,Silver,23,169.54,3899.42,273,Online Wallet,Android,Yes,7,4


##  حذف رکوردهای تکراری

In [ ]:
duplicates = df.duplicated().sum()
print(duplicates)
df = df.drop_duplicates()

0


## حذف فاصله‌های اضافه از متن‌ها

In [ ]:
string_cols = df.select_dtypes(include="object").columns
print(string_cols)
for col in string_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )


Index(['first_name', 'gender', 'city', 'province', 'signup_date',
       'membership_tier', 'payment_method', 'device', 'discount_used'],
      dtype='str')


C:\Users\Erfan\AppData\Local\Temp\ipykernel_13636\3444483201.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include="object").columns


## یکدست کردن متن‌ها

In [ ]:
df["gender"] = (
    df["gender"]
    .str.upper()
    .replace({
        "MALE": "M",
        "FEMALE": "F"
    })
)
for col in string_cols:
    if col == "gender":
        df[col] = (
            df[col]
            .str.upper()
            .replace({
                "MALE": "M",
                "FEMALE": "F"
            })
        )
    else:
        df[col] = df[col].str.title()


## درست کردن نوع داده‌ها

In [31]:
# لیست ستون‌هایی که باید عددی باشن
numeric_cols = [
    "age",
    "purchase_count",
    "avg_order_value",
    "total_spending",
    "last_purchase_days",
    "returned_items",
    "satisfaction_score"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["signup_date"] = pd.to_datetime(
    df["signup_date"],
    errors="coerce"
)


## حذف مقادیر غیرممکن

In [33]:
df.loc[
    (df.age < 15) | (df.age > 100),
    "age"
] = np.nan

for col in numeric_cols:
    df.loc[
        df[col] < 0,
        col
    ] = np.nan
    
df.loc[
    (df.satisfaction_score < 1) |
    (df.satisfaction_score > 5),
    "satisfaction_score"
] = np.nan


## شناسایی و حذف مقادیر پرت 

In [ ]:
for col in numeric_cols:
    Q1 = df[col].quantile(.25)
    Q3 = df[col].quantile(.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = (df[col] < lower) | (df[col] > upper)

    print(f"{col} {outliers.sum()}")
    
    df.loc[outliers, col] = np.nan


## مدیریت مقادیر خالی

In [29]:
print(df.isnull().sum())

customer_id           0
first_name            0
gender                0
age                   1
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64


In [32]:
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in string_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df["signup_date"] = df["signup_date"].fillna(
    df["signup_date"].median()
)


## هماهنگی بین ستون‌ها

In [35]:
mask = df["returned_items"] > df["purchase_count"]
print(mask)
print(mask.sum())

df.loc[mask, "returned_items"] = df.loc[mask, "purchase_count"]


0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
50    False
51    False
52    False
53    False
54    False
55    False
56    False
57    False
58    False
59    False
dtype: bool
0


In [36]:
expected = df["purchase_count"] * df["avg_order_value"]

difference = abs(expected - df["total_spending"])

mask = difference > expected * 0.05

print(mask.sum())

df.loc[mask, "total_spending"] = expected[mask]


2


## اصلاح غلط‌های تایپی رایج

In [ ]:
# اینجا باید داده اسم شهر های ایران را وارد کنیم و با اون تمام داده ها را پردازش کنیم ولی من نمیکنم
city_map = {
    "Teheran": "Tehran",
    "Tehran ": "Tehran",
    "Karadj": "Karaj"
}

province_map = {
    "Teheran": "Tehran",
    "Alburz": "Alborz"
}

df["city"] = df["city"].replace(city_map)
df["province"] = df["province"].replace(province_map)


## یکدست کردن فرمت تاریخ

In [38]:
df["signup_date"] = df["signup_date"].dt.strftime("%Y-%m-%d")

## ذخیره‌

In [ ]:
df.to_excel(
    "First Dataset_Cleaned.xlsx",
    index=False
)
